In [137]:
import re
import os

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Reading files from the directory

In [138]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## [Reading data] Creating a file_path variable for convenience 

In [139]:
file_path = "./input/for_analysis/results_synthetic/modelsize_memory/"

## [Reading data] Creating a file_name variable for convenience 

In [140]:
file_name="_modelsize_memory.csv"
file_name_quantized="_modelsize_memory_quantized.csv"

# Function for writing keys

In [141]:
def print_keys(key):
    latex_map = {
        "p1": r"$p1: \alpha_1(x) := \exists^{[8,10]}y\left(\alpha_0(y) \wedge \neg E(x,y)\right)$",
        "p2": r"$p2: \alpha_2(x) := \exists^{[10,30]}y\left(\alpha_1(y) \wedge \neg E(x,y)\right)$",
        "p3": r"$p3: \alpha_3(x) := \exists^{[10,30]}y\left(\alpha_2(y) \wedge \neg E(x,y)\right)$"
    }
    clean_latex = re.sub(r'[\x00-\x1F\x7F]', '', latex_map.get(key, ''))
    return clean_latex

# Model size

## [Model size]. Read the dataset and create a dataframe. Pattern:  df_acrgnn{activation_function}

In [142]:
df_acrgnn_relu=pd.read_csv(f'{file_path}acrgnn_relu{file_name}')

df_acrgnn_relu6=pd.read_csv(f'{file_path}acrgnn_relu6{file_name}')

df_acrgnn_trrelu=pd.read_csv(f'{file_path}acrgnn_trrelu{file_name}')

df_acrgnn_leakyrelu=pd.read_csv(f'{file_path}acrgnn_leakyrelu{file_name}')

df_acrgnn_gelu=pd.read_csv(f'{file_path}acrgnn_gelu{file_name}')

df_acrgnn_silu=pd.read_csv(f'{file_path}acrgnn_silu{file_name}')

df_acrgnn_sigmoid=pd.read_csv(f'{file_path}acrgnn_sigmoid{file_name}')

df_acrgnn_normalized=pd.read_csv(f'{file_path}acrgnn_normalized{file_name}')

df_acrgnn_softplus=pd.read_csv(f'{file_path}acrgnn_softplus{file_name}')

df_acrgnn_elu=pd.read_csv(f'{file_path}acrgnn_elu{file_name}')

In [143]:
df_acrgnn_relu_quantized=pd.read_csv(f'{file_path}acrgnn_relu{file_name_quantized}')

df_acrgnn_relu6_quantized=pd.read_csv(f'{file_path}acrgnn_relu6{file_name_quantized}')

df_acrgnn_trrelu_quantized=pd.read_csv(f'{file_path}acrgnn_trrelu{file_name_quantized}')

df_acrgnn_leakyrelu_quantized=pd.read_csv(f'{file_path}acrgnn_leakyrelu{file_name_quantized}')

df_acrgnn_gelu_quantized=pd.read_csv(f'{file_path}acrgnn_gelu{file_name_quantized}')

df_acrgnn_silu_quantized=pd.read_csv(f'{file_path}acrgnn_silu{file_name_quantized}')

df_acrgnn_sigmoid_quantized=pd.read_csv(f'{file_path}acrgnn_sigmoid{file_name_quantized}')

df_acrgnn_normalized_quantized=pd.read_csv(f'{file_path}acrgnn_normalized{file_name_quantized}')

df_acrgnn_softplus_quantized=pd.read_csv(f'{file_path}acrgnn_softplus{file_name_quantized}')

df_acrgnn_elu_quantized=pd.read_csv(f'{file_path}acrgnn_elu{file_name_quantized}')

In [144]:
df_acrgnn_relu.head()

,activation_function,key,layer,hidden_dimension,learning_rate,model_file_size_mb,total_checkpoint_MB,trainable_MB,bn_params_MB,bn_buffers_MB,trainable_params,bn_params
0,relu,p1,1,2,0.010,0.005966,0.000198,0.000175,0.000015,0.000023,46,4
1,relu,p1,1,2,0.001,0.005982,0.000198,0.000175,0.000015,0.000023,46,4
2,relu,p1,1,2,0.005,0.005982,0.000198,0.000175,0.000015,0.000023,46,4
3,relu,p1,2,2,0.010,0.009867,0.000305,0.000259,0.000031,0.000046,68,8
4,relu,p1,2,2,0.001,0.009893,0.000305,0.000259,0.000031,0.000046,68,8


In [145]:
dfs = [df_acrgnn_relu, df_acrgnn_relu6, df_acrgnn_trrelu, df_acrgnn_leakyrelu, df_acrgnn_gelu, df_acrgnn_silu, df_acrgnn_sigmoid, df_acrgnn_normalized, df_acrgnn_softplus, df_acrgnn_elu]

In [146]:
dfs_quantized = [df_acrgnn_relu_quantized, df_acrgnn_relu6_quantized, df_acrgnn_trrelu_quantized, df_acrgnn_leakyrelu_quantized, df_acrgnn_gelu_quantized, df_acrgnn_silu_quantized, df_acrgnn_sigmoid_quantized, df_acrgnn_normalized_quantized, df_acrgnn_softplus_quantized, df_acrgnn_elu_quantized]

In [147]:
activation_map = {
    "relu": "ReLU",
    "relu6": "ReLU6",
    "trrelu": "trReLU",
    "leakyrelu": "LeakyReLU",
    "gelu": "GELU",
    "silu": "SiLU",
    "sigmoid": "Sigmoid",
    "normalized": "Normalized",
    "softplus": "Softplus",
    "elu": "ELU"
}

In [148]:
for df in dfs:
    df["activation_function"] = df["activation_function"].replace(activation_map)

In [149]:
for df in dfs_quantized:
    df["activation_function"] = df["activation_function"].replace(activation_map)

In [150]:
df_acrgnn_relu.columns

Index(['activation_function', 'key', 'layer', 'hidden_dimension',
       'learning_rate', 'model_file_size_mb', 'total_checkpoint_MB',
       'trainable_MB', 'bn_params_MB', 'bn_buffers_MB', 'trainable_params',
       'bn_params'],
      dtype='object')

In [151]:
def combine_with_activation(dataframes):
    combined = []
    for df in dataframes:
        temp = df.copy()
        combined.append(temp[['activation_function', 'key', 'layer', 'hidden_dimension',
       'learning_rate', 'model_file_size_mb', 'total_checkpoint_MB',
       'trainable_MB', 'bn_params_MB', 'bn_buffers_MB', 'trainable_params',
       'bn_params']])
    return pd.concat(combined, ignore_index=True)

In [152]:
combined_acrgnn_activationFunction = combine_with_activation(dfs)
combined_acrgnn_activationFunction.head(500)

,activation_function,key,layer,hidden_dimension,learning_rate,model_file_size_mb,total_checkpoint_MB,trainable_MB,bn_params_MB,bn_buffers_MB,trainable_params,bn_params
0,ReLU,p1,1,2,0.010,0.005966,0.000198,0.000175,0.000015,0.000023,46,4
1,ReLU,p1,1,2,0.001,0.005982,0.000198,0.000175,0.000015,0.000023,46,4
2,ReLU,p1,1,2,0.005,0.005982,0.000198,0.000175,0.000015,0.000023,46,4
3,ReLU,p1,2,2,0.010,0.009867,0.000305,0.000259,0.000031,0.000046,68,8
4,ReLU,p1,2,2,0.001,0.009893,0.000305,0.000259,0.000031,0.000046,68,8
...,...,...,...,...,...,...,...,...,...,...,...,...
495,ReLU,p2,6,9,0.010,0.033892,0.007126,0.006668,0.000412,0.000458,1748,108
496,ReLU,p2,6,9,0.001,0.033960,0.007126,0.006668,0.000412,0.000458,1748,108
497,ReLU,p2,6,9,0.005,0.033960,0.007126,0.006668,0.000412,0.000458,1748,108
498,ReLU,p2,7,9,0.010,0.039257,0.008301,0.007767,0.000481,0.000534,2036,126


In [153]:
combined_acrgnn_activationFunction_quantized = combine_with_activation(dfs_quantized)
combined_acrgnn_activationFunction_quantized.head(500)

,activation_function,key,layer,hidden_dimension,learning_rate,model_file_size_mb,total_checkpoint_MB,trainable_MB,bn_params_MB,bn_buffers_MB,trainable_params,bn_params
0,ReLU,p1,1,2,0.010,0.010204,0.000084,0.000061,0.000015,0.000023,12,4
1,ReLU,p1,1,2,0.001,0.010228,0.000084,0.000061,0.000015,0.000023,12,4
2,ReLU,p1,1,2,0.005,0.010228,0.000084,0.000061,0.000015,0.000023,12,4
3,ReLU,p1,2,2,0.010,0.017553,0.000156,0.000111,0.000031,0.000046,22,8
4,ReLU,p1,2,2,0.001,0.017593,0.000156,0.000111,0.000031,0.000046,22,8
...,...,...,...,...,...,...,...,...,...,...,...,...
495,ReLU,p2,6,9,0.010,0.048481,0.001087,0.000629,0.000412,0.000458,146,108
496,ReLU,p2,6,9,0.001,0.048586,0.001087,0.000629,0.000412,0.000458,146,108
497,ReLU,p2,6,9,0.005,0.048586,0.001087,0.000629,0.000412,0.000458,146,108
498,ReLU,p2,7,9,0.010,0.056090,0.001266,0.000732,0.000481,0.000534,170,126


# Memory Reduction

Key findings> hidden_dimension=32, learning_rate=0.01, layer=2

memory_reduction=memory_reduction[(memory_reduction.hidden_dimension==32) & (memory_reduction.layer==2) & (memory_reduction.learning_rate==0.01)]

In [154]:
merge_cols = [
    'activation_function',
    'key',
    'layer',
    'hidden_dimension',
    'learning_rate'
]

memory_reduction = combined_acrgnn_activationFunction.merge(
    combined_acrgnn_activationFunction_quantized,
    on=merge_cols,
    suffixes=('_fp32', '_int8')
)
memory_reduction.head()

,activation_function,key,layer,hidden_dimension,learning_rate,model_file_size_mb_fp32,total_checkpoint_MB_fp32,trainable_MB_fp32,bn_params_MB_fp32,bn_buffers_MB_fp32,trainable_params_fp32,bn_params_fp32,model_file_size_mb_int8,total_checkpoint_MB_int8,trainable_MB_int8,bn_params_MB_int8,bn_buffers_MB_int8,trainable_params_int8,bn_params_int8
0,ReLU,p1,1,2,0.010,0.005966,0.000198,0.000175,0.000015,0.000023,46,4,0.010204,0.000084,0.000061,0.000015,0.000023,12,4
1,ReLU,p1,1,2,0.001,0.005982,0.000198,0.000175,0.000015,0.000023,46,4,0.010228,0.000084,0.000061,0.000015,0.000023,12,4
2,ReLU,p1,1,2,0.005,0.005982,0.000198,0.000175,0.000015,0.000023,46,4,0.010228,0.000084,0.000061,0.000015,0.000023,12,4
3,ReLU,p1,2,2,0.010,0.009867,0.000305,0.000259,0.000031,0.000046,68,8,0.017553,0.000156,0.000111,0.000031,0.000046,22,8
4,ReLU,p1,2,2,0.001,0.009893,0.000305,0.000259,0.000031,0.000046,68,8,0.017593,0.000156,0.000111,0.000031,0.000046,22,8


## Derive Core Metrics

Disk comression

In [155]:
memory_reduction['model_file_size_ratio'] = memory_reduction['model_file_size_mb_fp32'] / memory_reduction['model_file_size_mb_int8']

### Compression Ratio (Memory)

In [157]:
memory_reduction['compression_ratio'] = (memory_reduction['trainable_MB_fp32']/ memory_reduction['trainable_MB_int8'])

### Effective Bitwidth

In [158]:
memory_reduction['bytes_per_param_fp32'] = (memory_reduction['trainable_MB_fp32'] * 1024 * 1024) / memory_reduction['trainable_params_fp32']
memory_reduction['effective_bitwidth'] = memory_reduction['bytes_per_param_fp32'] * 8
memory_reduction['bytes_per_param_int8'] = (memory_reduction['trainable_MB_int8'] * 1024 * 1024) / memory_reduction['trainable_params_int8']
memory_reduction['effective_bitwidth_quantized'] = memory_reduction['bytes_per_param_int8'] * 8

### BN Fraction

In [159]:
memory_reduction['bn_fraction_fp32'] = memory_reduction['bn_params_MB_fp32'] / memory_reduction['trainable_MB_fp32']
memory_reduction['bn_fraction_int8'] = memory_reduction['bn_params_MB_int8'] / memory_reduction['trainable_MB_int8']

### Buffer Overhead

In [160]:
memory_reduction['buffer_fraction_fp32'] = memory_reduction['bn_buffers_MB_fp32'] / memory_reduction['trainable_MB_fp32']
memory_reduction['buffer_fraction_int8'] = memory_reduction['bn_buffers_MB_int8'] / memory_reduction['trainable_MB_int8']

In [161]:
memory_reduction

,activation_function,key,layer,hidden_dimension,learning_rate,model_file_size_mb_fp32,total_checkpoint_MB_fp32,trainable_MB_fp32,bn_params_MB_fp32,bn_buffers_MB_fp32,...,model_file_size_ratio,compression_ratio,bytes_per_param_fp32,effective_bitwidth,bytes_per_param_int8,effective_bitwidth_quantized,bn_fraction_fp32,bn_fraction_int8,buffer_fraction_fp32,buffer_fraction_int8
0,ReLU,p1,1,2,0.010,0.005966,0.000198,0.000175,0.000015,0.000023,...,0.584673,2.875000,4.0,32.0,5.333333,42.666667,0.086957,0.250000,0.130435,0.375000
1,ReLU,p1,1,2,0.001,0.005982,0.000198,0.000175,0.000015,0.000023,...,0.584895,2.875000,4.0,32.0,5.333333,42.666667,0.086957,0.250000,0.130435,0.375000
2,ReLU,p1,1,2,0.005,0.005982,0.000198,0.000175,0.000015,0.000023,...,0.584895,2.875000,4.0,32.0,5.333333,42.666667,0.086957,0.250000,0.130435,0.375000
3,ReLU,p1,2,2,0.010,0.009867,0.000305,0.000259,0.000031,0.000046,...,0.562099,2.344828,4.0,32.0,5.272727,42.181818,0.117647,0.275862,0.176471,0.413793
4,ReLU,p1,2,2,0.001,0.009893,0.000305,0.000259,0.000031,0.000046,...,0.562337,2.344828,4.0,32.0,5.272727,42.181818,0.117647,0.275862,0.176471,0.413793
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8995,ELU,p3,9,32,0.001,0.151568,0.113480,0.111214,0.002197,0.002266,...,1.499250,44.172727,4.0,32.0,4.177215,33.417722,0.019757,0.872727,0.020375,0.900000
8996,ELU,p3,9,32,0.005,0.151568,0.113480,0.111214,0.002197,0.002266,...,1.499250,44.172727,4.0,32.0,4.177215,33.417722,0.019757,0.872727,0.020375,0.900000
8997,ELU,p3,10,32,0.010,0.168196,0.126060,0.123543,0.002441,0.002518,...,1.493159,44.182810,4.0,32.0,4.176638,33.413105,0.019762,0.873124,0.020379,0.900409
8998,ELU,p3,10,32,0.001,0.168306,0.126060,0.123543,0.002441,0.002518,...,1.457975,44.182810,4.0,32.0,4.176638,33.413105,0.019762,0.873124,0.020379,0.900409


In [162]:
memory_reduction_best_values = memory_reduction[(memory_reduction.hidden_dimension==32) & (memory_reduction.layer==2) & (memory_reduction.learning_rate==0.01)]
memory_reduction_best_values

,activation_function,key,layer,hidden_dimension,learning_rate,model_file_size_mb_fp32,total_checkpoint_MB_fp32,trainable_MB_fp32,bn_params_MB_fp32,bn_buffers_MB_fp32,...,model_file_size_ratio,compression_ratio,bytes_per_param_fp32,effective_bitwidth,bytes_per_param_int8,effective_bitwidth_quantized,bn_fraction_fp32,bn_fraction_int8,buffer_fraction_fp32,buffer_fraction_int8
273,ReLU,p1,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
573,ReLU,p2,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
873,ReLU,p3,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
1173,ReLU6,p1,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
1473,ReLU6,p2,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
1773,ReLU6,p3,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
2073,trReLU,p1,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
2373,trReLU,p2,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
2673,trReLU,p3,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906
2973,LeakyReLU,p1,2,32,0.01,0.034918,0.025414,0.02491,0.000488,0.000504,...,1.454784,43.825503,4.0,32.0,4.197183,33.577465,0.019602,0.85906,0.020214,0.885906


In [163]:
memory_reduction_best_values['model_file_size_ratio'].unique() 

array([1.45478385])

In [164]:
memory_reduction_best_values['bytes_per_param_fp32'].unique(),memory_reduction_best_values['effective_bitwidth'].unique(),memory_reduction_best_values['bytes_per_param_int8'].unique(),memory_reduction_best_values['effective_bitwidth_quantized'].unique()

(array([4.]), array([32.]), array([4.1971831]), array([33.57746479]))